In [20]:
import requests
import pandas as pd
import numpy as np
import json
import re
import emoji
import nltk
from nltk.corpus import stopwords

# 1. Initial Inspection of the Dataset

In [2]:
url = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Cell_Phones_and_Accessories.jsonl"
records = []
with requests.get(url,stream=True) as r:
    for i,line in enumerate(r.iter_lines()):
        if i>200000:
            break
        if line:
            records.append(json.loads(line.decode("utf-8")))

df = pd.DataFrame(records)

## Data Loading — Issue & Fix

**Problem:**
The standard `load_dataset()` from HuggingFace failed because the Amazon Reviews 2023
dataset uses a custom loading script (`Amazon-Reviews-2023.py`) which is no longer
supported by newer versions of the `datasets` library.

**What we tried:**
- `load_dataset()` with `trust_remote_code=True` → blocked by library version
- `load_dataset()` with `streaming=True` → same error

**Fix:**
Bypassed HuggingFace loader entirely. Used `requests` to stream the raw `.jsonl` file
directly from the HuggingFace CDN, reading 200K rows line by line without downloading
the full 9.34GB file.

**Why streaming?**
The full Cell Phones & Accessories review file is 9.34GB. Streaming reads one line at a
time and stops at 200K rows — no unnecessary data downloaded.

**Why `json.loads()` instead of `pd.read_json()`?**
Each streamed line arrives as `bytes`. I decoded it to a string with `.decode("utf-8")`,
then parse it into a Python dict with `json.loads()`. `pd.read_json()` expects a file
path or file-like object, not a raw string.

In [3]:
print(df.shape)
print(df.columns)
print(df.head())

(200001, 10)
Index(['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase'],
      dtype='object')
   rating                                title  \
0     4.0     No white background! It’s clear!   
1     5.0  Awesome!  Great price!  Works well!   
2     5.0   Worked but took an hour to install   
3     4.0                               Decent   
4     5.0                             LOVE IT!   

                                                text  \
0  I bought this bc I thought it had the nice whi...   
1  Perfect. How pissed am I that I recently paid ...   
2  Overall very happy with the end result. If you...   
3  Lasted about 9 months then the lock button bro...   
4  LOVE THIS CASE! Works better than my expensive...   

                                              images        asin parent_asin  \
0  [{'small_image_url': 'https://images-na.ssl-im...  B08L6L3X1S  B08L6L3X1S   
1                              

In [4]:
df_filtered = df.drop(columns=['images','parent_asin','user_id',])

In [5]:
df_filtered = df_filtered.drop(df_filtered[df_filtered['rating']>2.0].index)
print("After rating filter: ",df_filtered.shape)

After rating filter:  (30197, 7)


In [6]:
df_filtered = df_filtered.dropna(subset=['text'])
print("After dropping Nulls: ", df_filtered.shape)

After dropping Nulls:  (30197, 7)


In [7]:
print(df_filtered['rating'].value_counts())
print(df_filtered['text'].str.split().str.len().describe())
print(df_filtered.head(3)[['rating','title','text']])

rating
1.0    19261
2.0    10936
Name: count, dtype: int64
count    30197.000000
mean        47.191906
std         62.406626
min          0.000000
25%         15.000000
50%         30.000000
75%         57.000000
max       1729.000000
Name: text, dtype: float64
    rating                                              title  \
16     2.0                       Don't try to tighten it up!!   
27     2.0  A bit complicated to apply. Requires spraying ...   
30     1.0                                     Buyer BEWARE!!   

                                                 text  
16  Putting it on a night stand drawer or top of h...  
27  A bit complicated to apply. Requires spraying ...  
30  This product is cheaply made, some of the part...  


In [8]:
print(df_filtered['text'][df_filtered['text'].str.split().str.len()==0].count())

14


In [9]:
df_filtered = df_filtered[df_filtered['text'].str.split().str.len()!=0]

In [10]:
print(df_filtered['text'].str.split().str.len().min())

1


# 2. Text Preprocessing Pipeline

In [24]:
def text_preprocessing(text,remove_stopwords=False):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+","",text)
    text = emoji.replace_emoji(text,replace="")
    special_character_pattern = r'[^a-zA-Z0-9\s]+'
    text = re.sub(special_character_pattern,"",text)
    text = re.sub(r"\s+"," ",text).strip()
    if remove_stopwords:
        stop_words = set(stopwords.words("english"))
        words = text.split(" ")
        words = [word for word in words if word not in stop_words]
        text = " ".join(words)
    
    return text

In [25]:
sample = df_filtered['text'].iloc[0]
sample

"Putting it on a night stand drawer or top of headboard.  Just don't try to tighten it up to where it doesn't slip off.  When the husband tightened his up it popped off and cracked the head on the base.  Hope they stand behind the product, we will see?"

In [26]:
print("Cleaned text (with stopwords): ",text_preprocessing(sample,remove_stopwords=False))
print("Cleaned text (without stopwords): ",text_preprocessing(sample,remove_stopwords=True))

Cleaned text (with stopwords):  putting it on a night stand drawer or top of headboard just dont try to tighten it up to where it doesnt slip off when the husband tightened his up it popped off and cracked the head on the base hope they stand behind the product we will see
Cleaned text (without stopwords):  putting night stand drawer top headboard dont try tighten doesnt slip husband tightened popped cracked head base hope stand behind product see


In [ ]:
df_filtered['text_clean'] = df_filtered['text'].apply(lambda x: text_preprocessing(x,remove_stopwords=True))
df_filtered['text_clean_full'] = df_filtered['text'].apply(lambda x: text_preprocessing(x,remove_stopwords=False))

In [28]:
print(df_filtered[['text','text_clean','text_clean_full']].head(3))

                                                 text  \
16  Putting it on a night stand drawer or top of h...   
27  A bit complicated to apply. Requires spraying ...   
30  This product is cheaply made, some of the part...   

                                           text_clean  \
16  putting night stand drawer top headboard dont ...   
27  bit complicated apply requires spraying adhesi...   
30  product cheaply made parts missing directions ...   

                                      text_clean_full  
16  putting it on a night stand drawer or top of h...  
27  a bit complicated to apply requires spraying a...  
30  this product is cheaply made some of the parts...  


In [29]:
df_filtered.to_csv("../data/cleaned/cleaned_review.csv",index=False)

### Two versions of cleaned text
| Column | Stopwords | Used for |
|--------|-----------|----------|
| `text_clean` | Removed | TF-IDF |
| `text_clean_full` | Kept | Sentence Transformers |

Stopword removal helps sparse models like TF-IDF but hurts dense models like sentence
transformers — removing "not" from "not working" damages meaning.

### Pipeline steps
1. Lowercase
2. Remove URLs
3. Remove emojis
4. Remove special characters
5. Normalize whitespace
6. Remove stopwords (optional, controlled by `remove_stopwords` parameter)

### Data quality
- Filtered to 1 and 2-star reviews — low-rated reviews proxy support tickets
- Dropped 14 empty string reviews (empty string ≠ null, not caught by `dropna`)
- Final corpus: 30,183 complaints